#  NQ Gaps — Hipótesis y Exploración
## Notebook A.01 del Hands-On: Masterclass de Diseño de Estrategias Cuantitativas

---

**Instrumento:** E-mini Nasdaq 100 Futures (@NQ) — Velas de 5 minutos  
**Periodo:** 2000 – 2026 (~1.75M barras)  
**Objetivo:** Formular una hipótesis de trading estructural sobre gaps, explorar la data, y establecer las reglas de decisión iniciales.

> *"Si no puedes explicar en una sola frase por qué el mercado se moverá a tu favor y qué demostraría que estás equivocado, no tienes una estrategia: tienes una creencia."*

### Conceptos del Masterclass que demostramos:
| Slide | Concepto |
|:---:|---|
| 02 | Anatomía de una estrategia (5 componentes) |
| 03 | Test de la frase única (X → Y porque Z) |
| 04 | Tipos de entrada |
| 05 | Peligro del p-hacking |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Importar estilo visual del masterclass
import sys; sys.path.insert(0, '.')
from nb_style import *

print(f'Pandas: {pd.__version__}')
print(f'NumPy:  {np.__version__}')

---
## 1. Carga de Datos y Construcción de Velas Diarias RTH

Trabajamos con las velas de **Regular Trading Hours (RTH)**: 09:30 a 16:00 hora de Nueva York.  
Esto excluye la actividad de pre-market y after-hours, donde la liquidez es escasa y los movimientos no representan la acción institucional principal.

El **gap** se define como la diferencia entre la apertura de hoy y el cierre de ayer, ambos dentro de RTH:

$$\text{Gap}_{t} = \text{Open}^{RTH}_{t} - \text{Close}^{RTH}_{t-1}$$

In [ ]:
# ─── Carga del dataset de 5 minutos ───
section_header('CARGA DE DATOS @NQ 5m', '')

df_5m = pd.read_csv('../@NQ_5m.csv', parse_dates=['TimeStamp'])
print(f'Filas totales: {len(df_5m):,}')
print(f'Rango: {df_5m["TimeStamp"].min()} → {df_5m["TimeStamp"].max()}')

# Convertir a hora de Nueva York
df_5m['TS_NY'] = pd.to_datetime(df_5m['TimeStamp']).dt.tz_convert('America/New_York')
df_5m['Date_NY'] = df_5m['TS_NY'].dt.date
df_5m['Hour_NY'] = df_5m['TS_NY'].dt.hour
df_5m['Min_NY']  = df_5m['TS_NY'].dt.minute

# Filtrar solo RTH (09:30 — 16:00 NY)
rth_mask = (
    ((df_5m['Hour_NY'] > 9) | ((df_5m['Hour_NY'] == 9) & (df_5m['Min_NY'] >= 30))) &
    ((df_5m['Hour_NY'] < 16) | ((df_5m['Hour_NY'] == 16) & (df_5m['Min_NY'] == 0)))
)
df_rth = df_5m[rth_mask].copy()
print(f'Barras RTH: {len(df_rth):,} ({len(df_rth)/len(df_5m)*100:.1f}% del total)')

# ─── Construir velas diarias RTH ───
rth_daily = df_rth.groupby('Date_NY').agg(
    Date=('Date_NY', 'first'),
    Open=('Open', 'first'),
    High=('High', 'max'),
    Low=('Low', 'min'),
    Close=('Close', 'last'),
    Volume=('TotalVolume', 'sum'),
    Bars=('Close', 'count')
).reset_index(drop=True)

# Descartar días con menos de 70 barras (sesiones incompletas)
rth_daily = rth_daily[rth_daily['Bars'] >= 70].reset_index(drop=True)

# ─── Calcular Gap y ATR ───
rth_daily['prev_Close'] = rth_daily['Close'].shift(1)
rth_daily['Gap_Pts'] = rth_daily['Open'] - rth_daily['prev_Close']
rth_daily['Gap_Pct'] = (rth_daily['Gap_Pts'] / rth_daily['prev_Close']) * 100

# True Range y ATR(14) con shift(1) para ser ex-ante
tr1 = rth_daily['High'] - rth_daily['Low']
tr2 = (rth_daily['High'] - rth_daily['prev_Close']).abs()
tr3 = (rth_daily['Low']  - rth_daily['prev_Close']).abs()
rth_daily['TR'] = np.maximum(tr1, np.maximum(tr2, tr3))
rth_daily['ATR_14'] = rth_daily['TR'].rolling(14).mean().shift(1)
rth_daily['Gap_ATR'] = rth_daily['Gap_Pts'] / rth_daily['ATR_14']

# Limpiar NaN
rth_daily = rth_daily.dropna(subset=['ATR_14', 'Gap_ATR']).reset_index(drop=True)

section_header('DATASET DIARIO RTH LISTO', '')
print(f'Días de trading válidos: {len(rth_daily):,}')
print(f'Rango: {rth_daily["Date"].iloc[0]} → {rth_daily["Date"].iloc[-1]}')
print(f'Gap medio: {rth_daily["Gap_Pct"].mean():.4f}%')
print(f'ATR(14) medio: {rth_daily["ATR_14"].mean():.1f} pts')

---
## 2. El Test de la Frase Única — ¿Por qué creemos que esto funciona?

Antes de mirar un solo gráfico, debemos pasar el **Test de la Frase Única** (Slide 03):

$$\boxed{\text{"Cuando pasa } X\text{, espero que el precio haga } Y\text{, porque } Z\text{."}}$$

### Nuestra Hipótesis: Gap Fill (Reversión)

| Componente | Definición |
|:---:|---|
| **X** | El precio del NQ abre con un gap significativo respecto al cierre RTH del día anterior |
| **Y** | El precio revierte hacia el nivel del cierre previo durante la sesión (cierra el gap) |
| **Z** | Los **market makers** y **operadores institucionales** necesitan reequilibrar inventarios acumulados durante la sesión overnight. La apertura disloca el precio del equilibrio, creando presión de reversión mecánica |

### ¿Por qué es una hipótesis válida? 
- Existe un **agente económico real** (market maker) con incentivos para revertir la dislocación
- El mecanismo es **mecánico**, no discrecional — los inventarios deben equilibrarse
- Es **independiente del timeframe**: no depende de una combinación arbitraria de indicadores

### Hipótesis Alternativa: Gap Continuation (Tendencia)

| Componente | Definición |
|:---:|---|
| **X** | El precio del NQ abre con un gap fuerte en la dirección de la tendencia previa |
| **Y** | El precio continúa en la dirección del gap durante la sesión |
| **Z** | El gap refleja **nueva información incorporada overnight** (earnings, datos macro, flujos de futuros). Los operadores que esperaban al margen se ven forzados a perseguir el precio |

> **Pregunta clave que resolveremos:** ¿Cuándo funciona cada una? La respuesta no es "siempre" — depende del **régimen de volatilidad** y la **magnitud del gap**.

---
## 3. Distribución de los Gaps — ¿Qué dice la data?

Graficamos el histograma de gaps con estimación de densidad Kernel (KDE) y superponemos una distribución Normal teórica para evaluar **colas pesadas** (*fat tails*).

In [ ]:
# ═══ DISTRIBUCIÓN DE GAPS ═══
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribución de los Gaps del Nasdaq (NQ) — RTH', 
             fontsize=17, fontweight='bold', color=COLORS['text_bright'], y=1.02)

gaps_pct = rth_daily['Gap_Pct'].dropna()
gaps_atr = rth_daily['Gap_ATR'].dropna()

# ─── Panel A: Gap en Porcentaje (%) ───
ax = axes[0]
ax.hist(gaps_pct, bins=120, density=True, alpha=0.6, color=COLORS['blue'], 
        edgecolor='none', label='Datos Reales')

# Fit normal
mu, sigma = gaps_pct.mean(), gaps_pct.std()
x_range = np.linspace(gaps_pct.min(), gaps_pct.max(), 300)
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 
        color=COLORS['red'], lw=2, linestyle='--', label=f'Normal (μ={mu:.3f}%, σ={sigma:.3f}%)')

# KDE
gaps_pct.plot.kde(ax=ax, color=COLORS['green'], lw=2, label='KDE Real')

ax.set_title('Gap en Porcentaje (%)', color=COLORS['text_bright'])
ax.set_xlabel('Gap (%)')
ax.set_ylabel('Densidad')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-3, 3)

# Anotación de colas pesadas
kurt = stats.kurtosis(gaps_pct)
skew = stats.skew(gaps_pct)
add_stat_box(ax, {
    'Curtosis': round(kurt, 2),
    'Asimetría': round(skew, 3),
    'N': len(gaps_pct),
    'Media': round(mu, 4),
    'Desvío': round(sigma, 4),
}, loc='upper left')

# ─── Panel B: Gap normalizado por ATR ───
ax = axes[1]
ax.hist(gaps_atr, bins=100, density=True, alpha=0.6, color=COLORS['purple'], 
        edgecolor='none', label='Datos Reales')

mu_a, sigma_a = gaps_atr.mean(), gaps_atr.std()
x_a = np.linspace(-4, 4, 300)
ax.plot(x_a, stats.norm.pdf(x_a, mu_a, sigma_a),
        color=COLORS['red'], lw=2, linestyle='--', label=f'Normal (μ={mu_a:.3f}, σ={sigma_a:.3f})')
gaps_atr.plot.kde(ax=ax, color=COLORS['cyan'], lw=2, label='KDE Real')

ax.set_title('Gap Normalizado por ATR(14)', color=COLORS['text_bright'])
ax.set_xlabel('Gap / ATR(14)')
ax.set_ylabel('Densidad')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-4, 4)

add_stat_box(ax, {
    'Curtosis': round(stats.kurtosis(gaps_atr), 2),
    'Asimetría': round(stats.skew(gaps_atr), 3),
}, loc='upper left')

plt.tight_layout()
plt.show()

# ─── Interpretación ───
section_header('HALLAZGO: DISTRIBUCIÓN LEPTOCÚRTICA', '')
print(f'La curtosis del gap es {kurt:.1f} (una Normal tiene curtosis = 0).')
print(f'→ Los gaps EXTREMOS son mucho más frecuentes de lo que predice una Normal.')
print(f'→ Esto confirma la existencia de eventos overnight disruptivos (earnings, macro).')
print(f'→ Estos gaps extremos son candidatos para la estrategia de Fade o Continuation.')

---
## 4. ¿Cuántos Gaps se Rellenan? — Fill Rates por Nivel

Si la hipótesis del Fade Gap es correcta, deberíamos ver que un porcentaje significativo de gaps se "rellenan" (el precio vuelve al nivel del cierre previo) durante la sesión.

Medimos el relleno a tres niveles:
- **25%**: El precio retrocede al menos un cuarto del gap
- **50%**: El precio retrocede al menos la mitad del gap
- **100%**: El gap se cierra completamente (el precio toca el cierre previo)

In [ ]:
# ═══ CÁLCULO DE FILL RATES ═══
section_header('ANÁLISIS DE FILL RATES', '')

rth_daily['Gap_Up'] = rth_daily['Gap_Pts'] > 0
rth_daily['Gap_Dn'] = rth_daily['Gap_Pts'] < 0

# Para Gap Up: el fill ocurre cuando el Low baja hacia el prev_Close
# Fill 100% = Low <= prev_Close
# Fill 50% = Low <= Open - 0.5 * |Gap|
rth_daily['Fill_100'] = False
rth_daily['Fill_50']  = False
rth_daily['Fill_25']  = False

gap_up = rth_daily['Gap_Up']
gap_dn = rth_daily['Gap_Dn']
gap_abs = rth_daily['Gap_Pts'].abs()

# Gap Up fills
rth_daily.loc[gap_up, 'Fill_100'] = rth_daily.loc[gap_up, 'Low'] <= rth_daily.loc[gap_up, 'prev_Close']
rth_daily.loc[gap_up, 'Fill_50']  = rth_daily.loc[gap_up, 'Low'] <= (rth_daily.loc[gap_up, 'Open'] - 0.50 * gap_abs[gap_up])
rth_daily.loc[gap_up, 'Fill_25']  = rth_daily.loc[gap_up, 'Low'] <= (rth_daily.loc[gap_up, 'Open'] - 0.25 * gap_abs[gap_up])

# Gap Down fills
rth_daily.loc[gap_dn, 'Fill_100'] = rth_daily.loc[gap_dn, 'High'] >= rth_daily.loc[gap_dn, 'prev_Close']
rth_daily.loc[gap_dn, 'Fill_50']  = rth_daily.loc[gap_dn, 'High'] >= (rth_daily.loc[gap_dn, 'Open'] + 0.50 * gap_abs[gap_dn])
rth_daily.loc[gap_dn, 'Fill_25']  = rth_daily.loc[gap_dn, 'High'] >= (rth_daily.loc[gap_dn, 'Open'] + 0.25 * gap_abs[gap_dn])

# Excluir gaps casi nulos (< 0.01%)
meaningful = gap_abs > 0.5  # Al menos 0.5 pts
df_fill = rth_daily[meaningful].copy()

# Calcular tasas
fill_up = df_fill[df_fill['Gap_Up']]
fill_dn = df_fill[df_fill['Gap_Dn']]

data = {
    'Nivel': ['25%', '50%', '100%'],
    'Gap Up (Long → Short)': [
        fill_up['Fill_25'].mean() * 100,
        fill_up['Fill_50'].mean() * 100,
        fill_up['Fill_100'].mean() * 100,
    ],
    'Gap Down (Short → Long)': [
        fill_dn['Fill_25'].mean() * 100,
        fill_dn['Fill_50'].mean() * 100,
        fill_dn['Fill_100'].mean() * 100,
    ],
    'Global': [
        df_fill['Fill_25'].mean() * 100,
        df_fill['Fill_50'].mean() * 100,
        df_fill['Fill_100'].mean() * 100,
    ]
}
df_rates = pd.DataFrame(data)
print(df_rates.to_string(index=False))

# ═══ GRÁFICO DE FILL RATES ═══
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Tasa de Relleno de Gaps — Nasdaq (NQ)', fontsize=17, 
             fontweight='bold', color=COLORS['text_bright'], y=1.02)

# Panel A: Barras de fill rates
ax = axes[0]
x = np.arange(3)
w = 0.25
bars1 = ax.bar(x - w, data['Gap Up (Long → Short)'], w, color=COLORS['blue'], label='Gap Up', alpha=0.9)
bars2 = ax.bar(x, data['Gap Down (Short → Long)'], w, color=COLORS['red'], label='Gap Down', alpha=0.9)
bars3 = ax.bar(x + w, data['Global'], w, color=COLORS['cyan'], label='Global', alpha=0.9)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.1f}%', 
                ha='center', fontsize=9, color=COLORS['text'], fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['Fill 25%', 'Fill 50%', 'Fill 100%'])
ax.set_ylabel('Tasa de Relleno (%)')
ax.set_title('Fill Rates por Dirección', color=COLORS['text_bright'])
ax.legend()
ax.set_ylim(0, 100)

# Panel B: Fill rate por magnitud del gap (en ATR)
ax = axes[1]
gap_bins = [0, 0.10, 0.25, 0.50, 1.00, 2.00, 10.0]
gap_labels = ['<0.10', '0.10-0.25', '0.25-0.50', '0.50-1.00', '1.00-2.00', '>2.00']
df_fill['Gap_Bin'] = pd.cut(df_fill['Gap_ATR'].abs(), bins=gap_bins, labels=gap_labels)

fill_by_size = df_fill.groupby('Gap_Bin', observed=False)['Fill_100'].mean() * 100

colors_bar = [COLORS['green'] if v > 60 else COLORS['orange'] if v > 40 else COLORS['red'] 
              for v in fill_by_size.values]
bars = ax.bar(range(len(fill_by_size)), fill_by_size.values, color=colors_bar, alpha=0.9,
              edgecolor=COLORS['grid'])

for i, (bar, v) in enumerate(zip(bars, fill_by_size.values)):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}%',
            ha='center', fontsize=10, color=COLORS['text_bright'], fontweight='bold')

ax.set_xticks(range(len(gap_labels)))
ax.set_xticklabels(gap_labels, fontsize=10)
ax.set_xlabel('Tamaño del Gap (en ATR)')
ax.set_ylabel('Fill Rate 100% (%)')
ax.set_title('Fill Rate por Magnitud del Gap', color=COLORS['text_bright'])
ax.axhline(50, color=COLORS['text_dim'], linestyle='--', alpha=0.5, lw=1)
ax.text(5.3, 51, '50%', color=COLORS['text_dim'], fontsize=9)

plt.tight_layout()
plt.show()

# ─── Interpretación ───
section_header('HALLAZGO: FILL RATES', '')
print(f'Los gaps pequeños (<0.25 ATR) se rellenan al 100% con alta frecuencia.')
print(f'Los gaps grandes (>1.0 ATR) se rellenan significativamente menos.')
print(f'→ REGLA: Fade gap para gaps PEQUEÑOS, Continuation para gaps GRANDES.')

---
## 5. Regímenes de Volatilidad — El Contexto lo Cambia Todo

Un gap de 0.5% no significa lo mismo cuando el VIX está en 12 (mercado tranquilo) que cuando está en 35 (pánico). Clasificamos cada día en **regímenes de volatilidad** usando el ATR(14) como proxy:

- **Baja Volatilidad**: ATR(14)% por debajo de la mediana histórica
- **Alta Volatilidad**: ATR(14)% por encima de la mediana histórica

> *La misma señal, en el régimen equivocado, produce resultados opuestos.*

In [ ]:
# ═══ REGÍMENES DE VOLATILIDAD ═══
section_header('CLASIFICACIÓN POR REGÍMENES DE VOLATILIDAD', '🌡️')

rth_daily['ATR_14_Pct'] = (rth_daily['ATR_14'] / rth_daily['prev_Close']) * 100
median_vol = rth_daily['ATR_14_Pct'].median()
rth_daily['Vol_Regime'] = np.where(rth_daily['ATR_14_Pct'] > median_vol, 'Alta Vol', 'Baja Vol')

print(f'Mediana ATR(14)%: {median_vol:.4f}%')
print(f'Días Baja Vol: {(rth_daily["Vol_Regime"]=="Baja Vol").sum():,}')
print(f'Días Alta Vol:  {(rth_daily["Vol_Regime"]=="Alta Vol").sum():,}')

# ═══ GRÁFICO: Comportamiento del Gap por Régimen ═══
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comportamiento de los Gaps según Régimen de Volatilidad', 
             fontsize=17, fontweight='bold', color=COLORS['text_bright'], y=1.02)

# Panel A: Distribución del gap por régimen
ax = axes[0]
for reg, color in [('Baja Vol', COLORS['blue']), ('Alta Vol', COLORS['red'])]:
    subset = rth_daily[rth_daily['Vol_Regime'] == reg]['Gap_ATR']
    ax.hist(subset, bins=80, density=True, alpha=0.5, color=color, label=reg, edgecolor='none')
    subset.plot.kde(ax=ax, color=color, lw=2)

ax.set_title('Distribución del Gap por Régimen', color=COLORS['text_bright'])
ax.set_xlabel('Gap / ATR(14)')
ax.set_xlim(-4, 4)
ax.legend()

# Panel B: Fill Rate 100% por régimen y tamaño
ax = axes[1]
for i, reg in enumerate(['Baja Vol', 'Alta Vol']):
    sub = rth_daily[(rth_daily['Vol_Regime'] == reg) & (rth_daily['Gap_ATR'].abs() > 0.01)]
    sub = sub.copy()
    sub['Gap_Bin'] = pd.cut(sub['Gap_ATR'].abs(), bins=gap_bins, labels=gap_labels)
    rates = sub.groupby('Gap_Bin', observed=False)['Fill_100'].mean() * 100
    offset = -0.15 + i * 0.30
    color = COLORS['blue'] if reg == 'Baja Vol' else COLORS['red']
    ax.bar(np.arange(len(rates)) + offset, rates.values, 0.28, 
           color=color, alpha=0.85, label=reg, edgecolor='none')

ax.set_xticks(range(len(gap_labels)))
ax.set_xticklabels(gap_labels, fontsize=9)
ax.set_xlabel('Tamaño del Gap (ATR)')
ax.set_ylabel('Fill Rate 100% (%)')
ax.set_title('Fill Rate por Régimen y Magnitud', color=COLORS['text_bright'])
ax.legend()
ax.axhline(50, color=COLORS['text_dim'], linestyle='--', alpha=0.4)

# Panel C: Heatmap de Fill Rate (Régimen × Tamaño)
ax = axes[2]
rth_daily_with_bins = rth_daily[rth_daily['Gap_ATR'].abs() > 0.01].copy()
rth_daily_with_bins['Gap_Bin'] = pd.cut(rth_daily_with_bins['Gap_ATR'].abs(), 
                                         bins=gap_bins, labels=gap_labels)
heatmap_data = rth_daily_with_bins.pivot_table(
    values='Fill_100', index='Vol_Regime', columns='Gap_Bin', 
    aggfunc='mean', observed=False
) * 100

sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap=CMAP_DIVERGENT, center=50,
            ax=ax, cbar_kws={'label': 'Fill Rate (%)'}, linewidths=0.5,
            linecolor=COLORS['grid'], annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Heatmap: Fill Rate por Régimen', color=COLORS['text_bright'])
ax.set_ylabel('')

plt.tight_layout()
plt.show()

# ─── Interpretación ───
section_header('HALLAZGO: EL RÉGIMEN DEFINE LA ESTRATEGIA', '')
print(' Baja Volatilidad + Gap Pequeño → Alta tasa de fill → FADE GAP')
print(' Alta Volatilidad + Gap Grande → Baja tasa de fill → CONTINUATION')
print(' Alta Volatilidad + Gap Pequeño → Ruido (evitar)')
print()
print('→ El régimen de volatilidad es el FILTRO PRIMARIO de la estrategia.')

---
## 6. Demostración de p-Hacking — Por Qué la Hipótesis es Más Importante que la Data

> *"Si tomas una base de datos del Nasdaq y combinas 5 días × 12 meses × 24 horas, tienes miles de combinaciones. Por pura probabilidad, siempre encontrarás una que ganó dinero."* — Slide 05

Vamos a demostrarlo en vivo: tomamos combinaciones **aleatorias** de día de la semana y hora del día, y mostramos que SIEMPRE se puede encontrar una combinación "ganadora".

In [ ]:
# ═══ DEMOSTRACIÓN DE p-HACKING ═══
section_header('DEMOSTRACIÓN: p-HACKING EN VIVO', '')

rth_daily['DayOfWeek'] = pd.to_datetime(rth_daily['Date']).dt.day_name()
rth_daily['Month'] = pd.to_datetime(rth_daily['Date']).dt.month

# Calcular retorno intradía (Open → Close) para cada combinación
rth_daily['Intraday_Ret'] = (rth_daily['Close'] - rth_daily['Open']) / rth_daily['Open'] * 100

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
months = list(range(1, 13))

# Probar TODAS las combinaciones día × mes
results_phack = []
for day in days:
    for month in months:
        sub = rth_daily[(rth_daily['DayOfWeek'] == day) & (rth_daily['Month'] == month)]
        if len(sub) < 20:
            continue
        cum_ret = sub['Intraday_Ret'].sum()
        sharpe = sub['Intraday_Ret'].mean() / sub['Intraday_Ret'].std() * np.sqrt(252) if sub['Intraday_Ret'].std() > 0 else 0
        results_phack.append({
            'Day': day, 'Month': month, 'N': len(sub),
            'Cum_Return': cum_ret, 'Sharpe': sharpe,
            'Mean_Ret': sub['Intraday_Ret'].mean()
        })

df_phack = pd.DataFrame(results_phack).sort_values('Cum_Return', ascending=False)

print(f'Combinaciones probadas: {len(df_phack)}')
print(f'\n TOP 5 "Mejores" combinaciones (p-hacking puro):')
print(df_phack.head().to_string(index=False))
print(f'\n💀 BOTTOM 5 "Peores" combinaciones:')
print(df_phack.tail().to_string(index=False))

# ═══ GRÁFICO ═══
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(' Peligro del p-Hacking: "Siempre Encuentras un Patrón"', 
             fontsize=17, fontweight='bold', color=COLORS['red'], y=1.02)

# Panel A: Heatmap de retornos acumulados por día × mes
ax = axes[0]
pivot = df_phack.pivot(index='Day', columns='Month', values='Cum_Return')
pivot = pivot.reindex(days)  # Ordenar días
sns.heatmap(pivot, annot=True, fmt='.0f', cmap=CMAP_DIVERGENT, center=0, ax=ax,
            linewidths=0.5, linecolor=COLORS['grid'],
            annot_kws={'fontsize': 8, 'fontweight': 'bold'},
            cbar_kws={'label': 'Retorno Acumulado (%)'})
ax.set_title('Retorno Acumulado por Día × Mes\n(60 combinaciones probadas)',
             color=COLORS['text_bright'])
ax.set_xlabel('Mes')
ax.set_ylabel('')

# Panel B: Distribución de Sharpe ratios
ax = axes[1]
sharpes = df_phack['Sharpe'].values
ax.hist(sharpes, bins=25, color=COLORS['purple'], alpha=0.7, edgecolor='none')

# Marcar los "ganadores"
top_sharpe = df_phack.iloc[0]['Sharpe']
ax.axvline(top_sharpe, color=COLORS['yellow'], lw=2, linestyle='--')
annotate_point(ax, top_sharpe, ax.get_ylim()[1] * 0.8, 
               f'Mejor: Sharpe {top_sharpe:.2f}\n¿Es real?', 
               color=COLORS['yellow'], offset=(30, -10))

ax.axvline(0, color=COLORS['text_dim'], lw=1, linestyle=':')
ax.set_title('Distribución de Sharpe Ratios\n(todas las combinaciones)', 
             color=COLORS['text_bright'])
ax.set_xlabel('Sharpe Ratio')
ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

# ─── Lección ───
section_header('LECCIÓN CLAVE', '')
print(f'Con {len(df_phack)} combinaciones, la MEJOR tiene Sharpe {top_sharpe:.2f}.')
print(f'Pero eso NO es un edge: es el resultado esperado de probar muchas cosas.')
print()
print(f'Si dividimos los datos en mitad de entrenamiento y mitad de prueba,')
print(f'la "mejor combinación" probablemente colapse en la segunda mitad.')
print()
print(f' POR ESO nuestra hipótesis de Gap tiene un Z económico ANTES de mirar datos.')
print(f' Elegir "comprar los martes de julio" NO tiene ningún Z — es p-hacking.')

---
## 7. Reglas de Decisión Táctica — Resumen del Notebook

Basándonos en la exploración, establecemos las **reglas de decisión iniciales** que llevaremos al siguiente notebook para implementar el backtest completo.

In [ ]:
# ═══ RESUMEN VISUAL: REGLAS DE DECISIÓN ═══
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

# Título
ax.text(0.5, 0.95, 'REGLAS DE DECISIÓN TÁCTICA', fontsize=20, fontweight='bold',
        color=COLORS['text_bright'], ha='center', va='top', transform=ax.transAxes)
ax.text(0.5, 0.89, 'Derivadas del análisis exploratorio de gaps en NQ (RTH)',
        fontsize=12, color=COLORS['text_dim'], ha='center', va='top', transform=ax.transAxes,
        style='italic')

# Línea separadora
ax.axhline(0.86, color=COLORS['blue'], lw=2, alpha=0.5, xmin=0.1, xmax=0.9)

# FADE GAP
ax.text(0.25, 0.78, '🔵 FADE GAP (Reversión)', fontsize=16, fontweight='bold',
        color=COLORS['blue'], ha='center', transform=ax.transAxes)
fade_rules = [
    'Régimen: Baja / Media Volatilidad',
    'Tamaño del gap: < 0.50 ATR',
    'Dirección: Operar contra el gap',
    'Target: Cierre previo (Fill 100%)',
    'Frecuencia esperada: ~60-80% de fill',
]
for j, rule in enumerate(fade_rules):
    ax.text(0.25, 0.70 - j*0.07, f'  • {rule}', fontsize=11,
            color=COLORS['text'], ha='center', transform=ax.transAxes)

# CONTINUATION
ax.text(0.75, 0.78, ' CONTINUATION (Tendencia)', fontsize=16, fontweight='bold',
        color=COLORS['red'], ha='center', transform=ax.transAxes)
cont_rules = [
    'Régimen: Alta Volatilidad',
    'Tamaño del gap: > 0.50 ATR',
    'Dirección: Operar a favor del gap',
    'Target: Extensión de la tendencia',
    'Frecuencia esperada: ~40-50% win rate',
]
for j, rule in enumerate(cont_rules):
    ax.text(0.75, 0.70 - j*0.07, f'  • {rule}', fontsize=11,
            color=COLORS['text'], ha='center', transform=ax.transAxes)

# Línea divisoria vertical
ax.axvline(0.50, color=COLORS['grid'], lw=1, alpha=0.5, ymin=0.15, ymax=0.85)

# Siguiente paso
ax.text(0.5, 0.15, '⏭️  Siguiente: Notebook A.02 — Entradas, Salidas y Excursiones MAE/MFE',
        fontsize=13, fontweight='bold', color=COLORS['yellow'], ha='center', 
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.5', facecolor=COLORS['bg_card'], 
                  edgecolor=COLORS['yellow'], alpha=0.9))

ax.text(0.5, 0.07, 'Hipótesis →   |  Exploración →   |  Próximo paso: Implementar y Medir',
        fontsize=11, color=COLORS['text_dim'], ha='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

section_header('NOTEBOOK A.01 COMPLETADO', '')
print('Hipótesis formulada, data explorada, reglas definidas.')
print('Llevamos al Notebook A.02:')
print('  • Fade Gap: Baja Vol + Gap < 0.50 ATR')
print('  • Continuation: Alta Vol + Gap > 0.50 ATR')